In [16]:
import pandas as pd
import matplotlib.pyplot as plt
import altair as alt
import numpy as np

In [124]:
'''
Load the label data
'''

label_df = pd.read_csv("labels.csv", index_col="ID")
seqs = label_df.index.tolist()
label_df

,Source,Isolation_site,Vancomycin_res,Ampicillin_res,Gentamicin_res,Linezolid_res,Year
ID,,,,,,,
50964202,Non-hospitalized person,Faeces,no,no,no,no,2015
50964205,Non-hospitalized person,Faeces,no,no,no,no,2015
T7EF-50964206,Non-hospitalized person,Faeces,no,no,no,no,2015
50964993,Non-hospitalized person,Faeces,no,no,no,no,2015
50964995,Non-hospitalized person,Faeces,no,no,no,no,2015
...,...,...,...,...,...,...,...
2020_975_2,Marine,Blue mussel,no,no,no,no,2020
2020_978,Marine,Blue mussel,no,no,no,no,2020
2020_979_1,Marine,Blue mussel,no,no,no,no,2020


In [125]:
'''
Make training, validation, and test splits
'''

train_frac = 0.6
val_frac   = 0.2
test_frac  = 0.2
print(train_frac, val_frac, test_frac)

n_train = round(train_frac * len(seqs))
n_val   = round(val_frac * len(seqs))
n_test  = round(test_frac * len(seqs))

print((n_train,n_test,n_val), n_train+n_test+n_val)

np.random.seed(168)
np.random.shuffle(seqs)

train_seqs = seqs[:n_train]
val_seqs = seqs[n_train:n_train+n_val]
test_seqs = seqs[-n_test:]

label_df.loc[train_seqs,"split"] = "training"
label_df.loc[val_seqs,"split"] = "validation"
label_df.loc[test_seqs,"split"] = "test"

label_df["split"].value_counts(dropna=False)

0.6 0.2 0.2
(772, 257, 257) 1286


training      772
validation    257
test          257
Name: split, dtype: int64

In [126]:
input_dropdown = alt.binding_select(options=label_df.columns.tolist()[:-1], name="Color")
col_param = alt.param(value="Year", bind=input_dropdown)

chart = alt.Chart(label_df).mark_bar().encode(
    y = alt.Y("sum(pct):Q").stack(None),
    # x = alt.X("x:N", title=""),
    # column = alt.Column("split:N", sort=["training","validation","test"])
    x = alt.X("split:N", sort=["training","validation","test"]),
    column = alt.Column("x:N", title="")
).transform_joinaggregate(
    total=f"count()",
    groupby=["split"]
).transform_calculate(
    x = f"datum[{col_param.name}]",
    pct = "1 / datum.total"
).add_params(
    col_param
)
# .transform_density(
#     "x",
#     as_=["x","fraction"],
#     groupby=["split"]
# )

chart

alt.Chart(...)

In [129]:
'''Save the data splits'''
split_df = pd.DataFrame(label_df["split"])
split_df.to_csv("sequence-splits.csv", index=True)
split_df

,split
ID,
50964202,validation
50964205,validation
T7EF-50964206,training
50964993,training
50964995,validation
...,...
2020_975_2,training
2020_978,test
2020_979_1,test
